# SANA Hyperparameter Search and Final Evaluation

Notebook for the full SANA experiment protocol:
- deterministic MS-COCO validation/test split;
- P1 hook-point search on validation;
- P2 alpha refinement on validation;
- final test comparison of best steering vs baseline vs random steering.

The notebook imports all reusable logic from `sana_experiment_utils.py` and uses `per_concept_bank` as the default SANA bank mode.


In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from sana_experiment_utils import (
    ALPHAS_P1,
    ALPHAS_P2_MASTER,
    DEFAULT_HOOK_POINTS,
    DEFAULT_SANA_BANK_MODE,
    build_refinement_window,
    download_coco_dataset,
    create_coco_split_manifests,
    get_device,
    load_sana_pipeline,
    run_test_evaluation,
    run_validation_sweep,
    smoke_test_setup,
    validate_split_manifests,
)

device = get_device()
print(f'Device: {device}')


In [ ]:
MODEL_ALIAS = 'small'
MODEL_ID = None
BANK_MODE = DEFAULT_SANA_BANK_MODE
COCO_DIR = 'coco'
EXPERIMENT_ROOT = 'sana_experiments'
STEERING_BANK_DIR = 'steering_vectors_sana'
SPLIT_SEED = 42
RANDOM_SV_SEED = 12345
IMAGES_PER_PROMPT = 5
VALIDATION_SIZE = 100
TEST_SIZE = 2000
NUM_DENOISING_STEPS = 20
NUM_CONCEPTS = 50
STRATEGY = "all_layers"
HOOK_POINTS_P1 = list(DEFAULT_HOOK_POINTS)
ALPHAS_P1_VALUES = list(ALPHAS_P1)
ALPHAS_P2_VALUES = list(ALPHAS_P2_MASTER)
RUN_SMOKE_TEST = False

exp_root = Path(EXPERIMENT_ROOT)
exp_root.mkdir(parents=True, exist_ok=True)
print('Experiment root:', exp_root.resolve())
print('Bank mode:', BANK_MODE)


In [ ]:
coco_info = download_coco_dataset(COCO_DIR)
coco_info


In [ ]:
splits = create_coco_split_manifests(
    coco_dir=COCO_DIR,
    output_dir=EXPERIMENT_ROOT,
    validation_size=VALIDATION_SIZE,
    test_size=TEST_SIZE,
    split_seed=SPLIT_SEED,
    seeds=list(range(IMAGES_PER_PROMPT)),
)

split_check = validate_split_manifests(
    validation_manifest=splits['validation_manifest'],
    test_manifest=splits['test_manifest'],
    validation_size=VALIDATION_SIZE,
    test_size=TEST_SIZE,
    images_per_prompt=IMAGES_PER_PROMPT,
)
split_check


In [ ]:
pipe = load_sana_pipeline(model_alias=MODEL_ALIAS, model_id=MODEL_ID, device=device)
print('Loaded model:', MODEL_ALIAS)
print('Transformer blocks:', len(pipe.transformer.transformer_blocks))

if RUN_SMOKE_TEST:
    smoke = smoke_test_setup(model_aliases=(MODEL_ALIAS,), hook_points=HOOK_POINTS_P1, device=device)
    smoke


## P1: Hook-point search on validation


In [ ]:
p1_root = exp_root / 'p1_validation_search'
p1_rows, p1_best = run_validation_sweep(
    pipe=pipe,
    validation_manifest=splits['validation_manifest'],
    validation_reference_dir=splits['validation_reference_dir'],
    output_root=str(p1_root),
    model_alias=MODEL_ALIAS,
    num_denoising_steps=NUM_DENOISING_STEPS,
    strategy=STRATEGY,
    num_concepts=NUM_CONCEPTS,
    hook_points=HOOK_POINTS_P1,
    alphas=ALPHAS_P1_VALUES,
    bank_mode=BANK_MODE,
    bank_dir=STEERING_BANK_DIR,
    device=device,
)
p1_df = pd.DataFrame(p1_rows).sort_values(["pareto_rank", "selection_score"], ascending=[True, False])
p1_df


In [ ]:
best_hook_point = p1_best["hook_point"]
best_alpha_p1 = p1_best["alpha"]
p2_alphas = build_refinement_window(best_alpha_p1, ALPHAS_P2_VALUES, window_size=5)
print('Best hook point from P1:', best_hook_point)
print('Best alpha from P1:', best_alpha_p1)
print('P2 refinement window:', p2_alphas)


## P2: Alpha refinement on validation


In [ ]:
p2_root = exp_root / 'p2_validation_refinement'
p2_rows, p2_best = run_validation_sweep(
    pipe=pipe,
    validation_manifest=splits['validation_manifest'],
    validation_reference_dir=splits['validation_reference_dir'],
    output_root=str(p2_root),
    model_alias=MODEL_ALIAS,
    num_denoising_steps=NUM_DENOISING_STEPS,
    strategy=STRATEGY,
    num_concepts=NUM_CONCEPTS,
    hook_points=[best_hook_point],
    alphas=p2_alphas,
    bank_mode=BANK_MODE,
    bank_dir=STEERING_BANK_DIR,
    device=device,
)
p2_df = pd.DataFrame(p2_rows).sort_values(["pareto_rank", "selection_score"], ascending=[True, False])
p2_df


In [ ]:
best_hook_point = p2_best["hook_point"]
best_alpha = p2_best["alpha"]
print('Selected best validation config:')
print(p2_best)


## P3: Final test comparison


In [ ]:
test_root = exp_root / 'p3_test_eval'
test_rows = run_test_evaluation(
    pipe=pipe,
    test_manifest=splits['test_manifest'],
    output_root=str(test_root),
    model_alias=MODEL_ALIAS,
    best_hook_point=best_hook_point,
    best_alpha=best_alpha,
    num_denoising_steps=NUM_DENOISING_STEPS,
    strategy=STRATEGY,
    num_concepts=NUM_CONCEPTS,
    bank_mode=BANK_MODE,
    bank_dir=STEERING_BANK_DIR,
    random_sv_seed=RANDOM_SV_SEED,
    device=device,
)
test_df = pd.DataFrame(test_rows)
test_df


In [ ]:
display_cols = [
    'variant', 'hook_point', 'alpha', 'clip_score_mean_mean', 'pick_score_mean_mean',
    'image_reward_mean_mean', 'mps_mean', 'vendi_score_mean'
]
test_df[display_cols].sort_values("variant")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

test_df.plot.bar(x="variant", y="clip_score_mean_mean", ax=axes[0], legend=False, title="CLIP Score")
test_df.plot.bar(x="variant", y="pick_score_mean_mean", ax=axes[1], legend=False, title="PickScore")
test_df.plot.bar(x="variant", y="vendi_score_mean", ax=axes[2], legend=False, title="Vendi")

for ax in axes:
    ax.grid(True, axis="y", alpha=0.3)
    ax.set_xlabel("")

plt.tight_layout()
plt.show()


## Optional P4

After P1-P3 are complete, you can add post-hoc experiments for `late_layers`, `early_layers`, or `timestep_scaled` by reusing the same functions with a different `STRATEGY`.
